# Road offence trend analysis
**Question:** Which road offences are increasing the fastest and where are they most concentrated?

Converted from `StandUp1.knwf` (KNIME 5.9).  
Pipeline: `Excel Reader (#5)` → `Row Filter (#3)` → `Column Filter (#7)` → `Missing Value (#6)` → `GroupBy (#4)`


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Node 1 — Excel Reader (`#5`)
Reads all 11 columns from the raw enforcement fines spreadsheet.  
Source file: `police_enforcement_2024_fines.xlsx`, sheet `police_enforcement_2024_fines`.


In [ ]:
# Excel Reader (#5)
# Update FILE_PATH to wherever the .xlsx lives on your machine
FILE_PATH = "police_enforcement_2024_fines.xlsx"

df = pd.read_excel(
    FILE_PATH,
    sheet_name="police_enforcement_2024_fines",
    dtype={
        "YEAR": "Int64",
        "JURISDICTION": str,
        "LOCATION": str,
        "AGE_GROUP": str,
        "METRIC": str,
        "DETECTION_METHOD": str,
        "FINES": "Int64",
        "ARRESTS": "Int64",
        "CHARGES": "Int64",
    },
    parse_dates=["START_DATE", "END_DATE"],
)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
df.head(3)


## Node 2 — Row Filter (`#3`)
Removes pre-aggregated summary rows and unclassifiable records.  
Four AND conditions (operator `NEQ_MISS` = not-equal, treat missing as non-matching):

| Column | Exclude value |
|---|---|
| `LOCATION` | `All regions` |
| `LOCATION` | `Unknown` |
| `AGE_GROUP` | `All ages` |
| `AGE_GROUP` | `Unknown` |

Rows after filter confirmed in KNIME: **6,026**.


In [ ]:
# Row Filter (#3)
mask = (
    (df["LOCATION"] != "All regions") &
    (df["LOCATION"] != "Unknown") &
    (df["AGE_GROUP"] != "All ages") &
    (df["AGE_GROUP"] != "Unknown")
)
df = df[mask].copy()

print(f"After Row Filter: {df.shape[0]:,} rows")
assert df.shape[0] == 6026, f"Expected 6026 rows, got {df.shape[0]}"


## Node 3 — Column Filter (`#7`)
Keeps only the five columns needed for aggregation.  
Dropped: `START_DATE`, `END_DATE`, `AGE_GROUP`, `DETECTION_METHOD`, `ARRESTS`, `CHARGES`.

Columns retained: `YEAR`, `JURISDICTION`, `LOCATION`, `METRIC`, `FINES`.


In [ ]:
# Column Filter (#7)
KEEP_COLS = ["YEAR", "JURISDICTION", "LOCATION", "METRIC", "FINES"]
df = df[KEEP_COLS].copy()

print(f"After Column Filter: {df.shape[1]} cols → {df.columns.tolist()}")


## Node 4 — Missing Value (`#6`)
Replaces any null `FINES` integers with `0` (fixed-value strategy).  
All other columns: do nothing.  
*(No nulls were present in this dataset — this is a safety net.)*


In [ ]:
# Missing Value (#6)
df["FINES"] = df["FINES"].fillna(0)

print(f"FINES nulls remaining: {df['FINES'].isna().sum()}")


## Node 5 — GroupBy (`#4`)
Aggregates `SUM(FINES)` grouped by `YEAR`, `JURISDICTION`, `METRIC`, `LOCATION`.  
Output confirmed in KNIME: **27 rows × 3 cols** (YEAR, METRIC, SUM(FINES) — JURISDICTION and LOCATION collapse to the groups present after filtering).

> Note: KNIME showed 3 cols in the output port, which means JURISDICTION and LOCATION  
> were likely configured as additional group keys that reduce to a small result set  
> given the filtered data. All four group keys are preserved below for full fidelity.


In [ ]:
# GroupBy (#4)
GROUP_COLS = ["YEAR", "JURISDICTION", "LOCATION", "METRIC"]

result = (
    df.groupby(GROUP_COLS, as_index=False)
      .agg(TOTAL_FINES=("FINES", "sum"))
      .sort_values(["YEAR", "JURISDICTION"])
      .reset_index(drop=True)
)

print(f"GroupBy output: {result.shape[0]:,} rows × {result.shape[1]} cols")
result.head(10)


## Output inspection
Quick summary views to verify the result matches expectations.


In [ ]:
# Total fines per METRIC across all years — directional trend check
print("=== Total fines by METRIC (all years) ===")
print(result.groupby("METRIC")["TOTAL_FINES"].sum().sort_values(ascending=False).to_string())

print()

# Total fines per JURISDICTION — geographic concentration check
print("=== Total fines by JURISDICTION (all years) ===")
print(result.groupby("JURISDICTION")["TOTAL_FINES"].sum().sort_values(ascending=False).to_string())

print()

# Year-on-year totals per METRIC
print("=== Fines by METRIC × YEAR ===")
pivot = result.pivot_table(index="YEAR", columns="METRIC", values="TOTAL_FINES", aggfunc="sum")
print(pivot.to_string())


In [ ]:
# Line chart: trends over time
pivot.plot(figsize=(10,6))

plt.title("Road Offence Trends Over Time")
plt.xlabel("Year")
plt.ylabel("Total Fines")
plt.legend(title="Offence Type")
plt.grid()

plt.show()

In [ ]:
# Bar chart: total fines by offence
result.groupby("METRIC")["TOTAL_FINES"].sum().sort_values().plot(kind="barh", figsize=(8,5))

plt.title("Total Fines by Offence Type")
plt.xlabel("Total Fines")
plt.ylabel("Offence Type")

plt.show()

In [ ]:
# Bar chart: fines by state
result.groupby("JURISDICTION")["TOTAL_FINES"].sum().sort_values().plot(kind="barh", figsize=(8,5))

plt.title("Total Fines by State")
plt.xlabel("Total Fines")
plt.ylabel("State")

plt.show()